# xgap: demo replay (step 2)

Thin by design -- no logic lives here. Mount Drive, clone/pull the repo,
run `setup_colab.sh`, call `scripts/run_demo_replay.py`. All control flow
lives in `xgap_code/` and `scripts/`; this notebook only sequences calls to
it.

Repo: https://github.com/AITEAM444/xgap (public)

**Run cell 1 before importing anything else, in every fresh runtime.** Colab's
`!` shell subprocess env does not propagate into this kernel's own Python
process, so `setup_colab.sh` cannot set `MUJOCO_GL` for you here -- see that
script's own comments.

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

XGAP_DRIVE_ROOT = "/content/drive/MyDrive/xgap"
XGAP_REPO_URL = "https://github.com/AITEAM444/xgap.git"

## Get the code

Clones into Drive on first run, `git pull`s on every run after -- code/config
history stays on GitHub (the source of truth), Drive is just where it lives so
`setup_colab.sh` / scripts can read it and outputs can be written next to it.
No manual re-uploading of files to Drive after this point; push to GitHub and
re-run this cell instead.

In [ ]:
if os.path.isdir(f"{XGAP_DRIVE_ROOT}/.git"):
    # Re-point + fast-forward rather than a bare `git pull`: a `.git` folder can already
    # exist here from an earlier manual Drive upload (predating the GitHub remote), which
    # has no upstream tracking and makes `git pull` fail with "no tracking information".
    # This folder is meant to be a pure mirror of GitHub with no local edits, so resetting
    # to origin's default branch is safe and sidesteps that class of stale-state error.
    !git -C {XGAP_DRIVE_ROOT} remote set-url origin {XGAP_REPO_URL}
    !git -C {XGAP_DRIVE_ROOT} fetch origin
    !git -C {XGAP_DRIVE_ROOT} checkout -B master origin/master
else:
    !git clone {XGAP_REPO_URL} {XGAP_DRIVE_ROOT}

## Environment rebuild

Idempotent -- safe to re-run. This also appends `nproc` / `nvidia-smi` /
`free -g` / installed library versions to `logs/env_meta.log` (Colab hardware
varies session to session), which satisfies the "log hardware in the first
cell" requirement without duplicating that logic here.

If this prints a restart banner, use *Runtime > Restart session* and re-run
this cell once (it will no-op on the already-satisfied install step) before
continuing.

In [ ]:
!bash {XGAP_DRIVE_ROOT}/setup_colab.sh

## Adopt resolved env vars into this kernel

`setup_colab.sh`'s own `export`s (`HF_HOME`, `HF_LEROBOT_HOME`,
`LIBERO_CONFIG_PATH`) only apply to ITS OWN subprocesses -- they vanish once
that script exits, same as any other `!`-cell `export`. Without this cell,
the next cell (a separate `!python ...` subprocess) sees none of them: caches
silently fall back to their unconfigured defaults, and `libero`'s interactive
dataset-path prompt reappears even though it already passed inside
`setup_colab.sh`. Reads `/content/.xgap_env` (written by the script above) into
`os.environ` in the KERNEL process instead, which every `!` cell for the rest
of this session DOES inherit -- same mechanism as why cell 1's `MUJOCO_GL`
works.

In [ ]:
with open("/content/.xgap_env") as f:
    for line in f:
        key, _, value = line.strip().partition("=")
        if key:
            os.environ[key] = value

for _k in ["XGAP_DRIVE_ROOT", "MUJOCO_GL", "HF_HOME", "HF_LEROBOT_HOME", "LIBERO_CONFIG_PATH"]:
    print(f"{_k}={os.environ.get(_k)}")

## Smoke run first

`configs/demo_replay_smoke.yaml`: 1 suite, 1 task, up to 5 episodes, both
`control_mode`s, with per-episode `.mp4` + trajectory plots saved locally
(`outputs/demo_replay_smoke/_local/videos/`) for visual inspection. See the
repo README, "How to read the first real run", for how to tell a crash apart
from an actual low-success-rate finding.

If re-running after a code change to the replay/logging/video pipeline
itself (not just a config change), clear old results first -- resume only
checks "does this episode's result file already exist", not whether it has
the fields/videos the current code would produce:

```python
!rm -rf /content/outputs/demo_replay_smoke
!rm -rf /content/drive/MyDrive/xgap/outputs/demo_replay_smoke
```

In [ ]:
!python {XGAP_DRIVE_ROOT}/scripts/run_demo_replay.py \
    --config {XGAP_DRIVE_ROOT}/configs/demo_replay_smoke.yaml

## control_freq comparison (done -- kept for reference)

`configs/demo_replay_smoke_cf20.yaml` -- same task/episode/control_modes as
the smoke config above, `control_freq=20` instead of `10`. This was run:
`control_freq=20` is now confirmed correct (see README "control_freq was
wrong from the start" -- LIBERO's own source shows demos are collected and
replayed 1:1 at 20Hz; `demo_replay_smoke.yaml`'s default was corrected to
match) and gets the eef position trajectory to track the recorded demo
almost exactly. But grasping still fails at `control_freq=20` too --
`gripper_qpos` closes fully (nothing between the fingers) both times, where
the recorded demo shows a partial close (something between the fingers)
both times. So this cell is no longer diagnostic on its own -- kept so the
comparison is reproducible -- and the investigation moved to the two cells
below instead.</cell id="b92b2689">


In [ ]:
!python {XGAP_DRIVE_ROOT}/scripts/run_demo_replay.py \
    --config {XGAP_DRIVE_ROOT}/configs/demo_replay_smoke_cf20.yaml

## Sanity check against a KNOWN answer (bisects the whole problem space in one shot)

None of this project's own code runs in this cell -- just `lerobot`'s own
`lerobot-eval` CLI against `lerobot/pi05_libero_finetuned`, a checkpoint with
a published official number: **96% on LIBERO-10** (see
`docs/source/libero.mdx` in lerobot, "Reproducing published results"). No
50GB dataset download needed either -- eval only touches the environment and
the (small) policy checkpoint.

- **~96% (or close)** -> installation, MuJoCo, success-checking, and
  init-state selection are all fine at the infrastructure level. Any
  remaining bug is specific to *this project's own harness* -- narrows
  straight to `within_task_index` (the init-state sweep below) or the raw
  action-injection path.
- **~0%** -> the environment layer itself is broken, independent of
  anything in `xgap_code/`. No amount of fixing our own harness would show
  up as a result until this is fixed first.

`n_episodes=5` for a quick, cheap signal -- raise it if the result is
ambiguous. This uses a real policy (not our demo-replay actions), so this is
also the first cell in this notebook that actually exercises the GPU.

In [ ]:
!lerobot-eval \
    --policy.path=lerobot/pi05_libero_finetuned \
    --policy.n_action_steps=10 \
    --env.type=libero \
    --env.task=libero_10 \
    --eval.batch_size=1 \
    --eval.n_episodes=5 \
    --env.max_parallel_tasks=1

## init_state sweep (is `within_task_index` picking the wrong index?)

`configs/sweep_init_states.yaml`: fixes ONE demo's recorded actions
(`libero_10` task 0, dataset episode_index=8 -- the same episode used in the
comparisons above) and replays them against **every** candidate `init_state`
LIBERO has for this task (`scripts/sweep_init_states.py`, no new
instrumentation -- one loop over `harness.get_num_init_states()`). Cheap
relative to another position/orientation investigation: same ~300-step
episode run N times (N = however many init_states this task has, LIBERO's
own count, not assumed), no additional download.

- **Any index succeeds** -> `within_task_index` (`dataset_io.py`)
  is picking the wrong LIBERO init_state, confirmed, and this sweep hands
  you the *correct* index directly (`sweep_summary.json`'s
  `successful_init_states`).
- **None succeed** -> init-state indexing is exonerated. Orientation
  (`eef_quat`, not currently in `state_chunk`) becomes the next thing to
  add and check -- see README.

In [ ]:
!python {XGAP_DRIVE_ROOT}/scripts/sweep_init_states.py \
    --config {XGAP_DRIVE_ROOT}/configs/sweep_init_states.yaml

## Full demo replay

Only run this after the smoke cell above completes cleanly (prints a
`decision` JSON, not a traceback).

In [ ]:
!python {XGAP_DRIVE_ROOT}/scripts/run_demo_replay.py \
    --config {XGAP_DRIVE_ROOT}/configs/demo_replay.yaml